# de Pablo Initial Latent To P-Ratio Correlation

Focused 04d. This notebook looks only at de Pablo and asks how the initial latent coordinates relate to final p-ratio.

It uses CV1 and CV2 only. For CV2, `z0 + z1` is not treated as the answer; instead the notebook fits a linear combination of `z0,z1` on validation networks, reports its weights, and evaluates it on test networks. The final section probes what each coordinate encodes using p-ratio and simple deformation-relevant network descriptors.



In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import torch

from lss.latent.analysis import (
    CVAnalysisContext, evaluate_readouts, fit_linear, path_curvature_metrics,
    pearson_r, predict_linear, r2_score, residualize,
)
from lss.utils import resolve_device


PLOT_COLORS = {
    'white': '#FFFFFF',
    'soft': '#F1ECEC',
    'muted': '#626456',
    'olive': '#545B4C',
    'deep': '#364735',
}
PLOT_CMAP = LinearSegmentedColormap.from_list(
    'paper_olive',
    [PLOT_COLORS['soft'], PLOT_COLORS['muted'], PLOT_COLORS['deep']],
)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 220,
    'font.size': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'figure.facecolor': PLOT_COLORS['soft'],
    'axes.facecolor': PLOT_COLORS['soft'],
    'axes.edgecolor': PLOT_COLORS['deep'],
    'axes.labelcolor': PLOT_COLORS['deep'],
    'xtick.color': PLOT_COLORS['deep'],
    'ytick.color': PLOT_COLORS['deep'],
    'grid.color': PLOT_COLORS['soft'],
    'text.color': PLOT_COLORS['deep'],
})









In [ ]:
# Config: keep this narrow while we build the story.

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('Could not locate project root containing pyproject.toml.')

cfg = {
    'dataset_names': ['depablo'],
    'latent_dims': [1, 2],
    'train_networks': 2,
    'train_frames_per_network': 2,
    'repeat_idx': 3,
    'target_mode': 'delta',
    'model_dir': ROOT / 'notebooks' / 'results' / 'latent_space_capacity_sweep' / 'full_sweep' / 'models',
    'device': 'auto',
    'pos_dim': 2,
    'ae_recon_max_frames_per_test_network': 15,
    'trajectory_max_frames_per_test_network': 100,
}

device = resolve_device(cfg['device'])
print('device:', device)
print('model dir:', cfg['model_dir'])
print('model choice:', f"nets={cfg['train_networks']}, frames={cfg['train_frames_per_network']}, rep={cfg['repeat_idx']}")


In [ ]:
# Shared saved-model and trajectory analysis context.
analysis = CVAnalysisContext(cfg=cfg, device=device, project_root=ROOT)
model_path = analysis.model_path
load_bundle = analysis.load_bundle
restore_ae = analysis.restore_ae
resolve_bundle_splits = analysis.resolve_bundle_splits
encode_frame_z = analysis.encode_frame_z
encode_initial_z = analysis.encode_initial_z
p_ratio_final = analysis.p_ratio_final
network_descriptor_row = analysis.network_descriptor_row
frame_deformation_row = analysis.frame_deformation_row


In [ ]:
# Build a compact table of initial latent coordinates on val and test networks.

rows = []
missing = []
for dataset_name in cfg['dataset_names']:
    for latent_dim in cfg['latent_dims']:
        path = model_path(dataset_name, latent_dim)
        if not path.exists():
            missing.append(path.name)
            continue
        print('loading', path.name)
        bundle = load_bundle(path)
        ae = restore_ae(bundle)
        _, val_data, test_data, _ = resolve_bundle_splits(bundle)
        for split_name, sims in [('val', val_data), ('test', test_data)]:
            for local_idx, sim in enumerate(sims):
                z = encode_initial_z(ae, bundle, sim)
                row = {
                    'dataset_name': dataset_name,
                    'latent_dim': int(latent_dim),
                    'split': split_name,
                    'sim_idx': int(local_idx),
                    'final_p_ratio': p_ratio_final(sim),
                    **network_descriptor_row(sim),
                }
                for dim_idx, value in enumerate(z):
                    row[f'z{dim_idx}'] = float(value)
                rows.append(row)
        del ae, bundle
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

latent_df = pd.DataFrame(rows)
if missing:
    print('missing models:', missing)
print('latent rows:', latent_df.shape)
display(latent_df.head())






In [ ]:
# Fit readouts on validation networks and score them on test networks.
metric_parts = []
weight_parts = []
pred_parts = []
for (dataset_name, latent_dim), group in latent_df.groupby(['dataset_name', 'latent_dim'], sort=False):
    metrics, weights = evaluate_readouts(group)
    metrics.insert(0, 'latent_dim', int(latent_dim))
    metrics.insert(0, 'dataset_name', dataset_name)
    weights.insert(0, 'latent_dim', int(latent_dim))
    weights.insert(0, 'dataset_name', dataset_name)
    metric_parts.append(metrics)
    weight_parts.append(weights)

    val = group[group['split'].eq('val')].copy()
    test = group[group['split'].eq('test')].copy()
    cols = ['z0'] if int(latent_dim) == 1 else ['z0', 'z1']
    model = fit_linear(val[cols].to_numpy(float), val['final_p_ratio'].to_numpy(float))
    test = test.copy()
    test['linear_combo_pred_p_ratio'] = predict_linear(model, test[cols].to_numpy(float))
    pred_parts.append(test)

metrics_df = pd.concat(metric_parts, ignore_index=True)
weights_df = pd.concat(weight_parts, ignore_index=True)
prediction_df = pd.concat(pred_parts, ignore_index=True)

print('Readout performance:')
display(metrics_df.round(4))
print('Linear readout weights fitted on validation networks:')
display(weights_df.round(5))





In [ ]:
# Scatter plots: test networks only. Lines/predictions are fitted on validation networks.

fig, axes = plt.subplots(len(cfg['dataset_names']), 3, figsize=(10.8, 3.2 * len(cfg['dataset_names'])), squeeze=False)

for row_idx, dataset_name in enumerate(cfg['dataset_names']):
    # CV1: z0 directly.
    ax = axes[row_idx, 0]
    group = latent_df[(latent_df['dataset_name'].eq(dataset_name)) & (latent_df['latent_dim'].eq(1))]
    if group.empty:
        ax.text(0.5, 0.5, 'missing CV1', transform=ax.transAxes, ha='center', va='center')
    else:
        val = group[group['split'].eq('val')]
        test = group[group['split'].eq('test')]
        model = fit_linear(val[['z0']].to_numpy(float), val['final_p_ratio'].to_numpy(float))
        x = test['z0'].to_numpy(float)
        y = test['final_p_ratio'].to_numpy(float)
        pred = predict_linear(model, x)
        order = np.argsort(x)
        ax.scatter(x, y, s=26, alpha=0.8, color=PLOT_COLORS['deep'])
        ax.plot(x[order], pred[order], color=PLOT_COLORS['deep'], lw=1.5)
        ax.set_title(f'{dataset_name} CV1: z0\nval-fit test R2={r2_score(y, pred):.3f}')
        ax.set_xlabel('z0')
        ax.set_ylabel('final p-ratio')

    # CV2: individual coordinates, not as the final p-ratio predictor.
    ax = axes[row_idx, 1]
    group = latent_df[(latent_df['dataset_name'].eq(dataset_name)) & (latent_df['latent_dim'].eq(2)) & (latent_df['split'].eq('test'))]
    if group.empty:
        ax.text(0.5, 0.5, 'missing CV2', transform=ax.transAxes, ha='center', va='center')
    else:
        y = group['final_p_ratio'].to_numpy(float)
        ax.scatter(group['z0'], y, s=24, alpha=0.75, label=f'z0 r={pearson_r(group["z0"], y):.2f}', color=PLOT_COLORS['deep'])
        ax.scatter(group['z1'], y, s=24, alpha=0.75, label=f'z1 r={pearson_r(group["z1"], y):.2f}', color=PLOT_COLORS['muted'])
        ax.set_title(f'{dataset_name} CV2: coordinates')
        ax.set_xlabel('coordinate value')
        ax.set_ylabel('final p-ratio')
        ax.legend(frameon=False)

    # CV2: fitted linear combination.
    ax = axes[row_idx, 2]
    pred = prediction_df[(prediction_df['dataset_name'].eq(dataset_name)) & (prediction_df['latent_dim'].eq(2))]
    if pred.empty:
        ax.text(0.5, 0.5, 'missing CV2', transform=ax.transAxes, ha='center', va='center')
    else:
        x = pred['linear_combo_pred_p_ratio'].to_numpy(float)
        y = pred['final_p_ratio'].to_numpy(float)
        r2 = r2_score(y, x)
        r = pearson_r(x, y)
        lim_min = min(np.nanmin(x), np.nanmin(y))
        lim_max = max(np.nanmax(x), np.nanmax(y))
        pad = 0.05 * (lim_max - lim_min + 1e-12)
        ax.scatter(x, y, s=26, alpha=0.8, color=PLOT_COLORS['olive'])
        ax.plot([lim_min - pad, lim_max + pad], [lim_min - pad, lim_max + pad], color=PLOT_COLORS['deep'], lw=1.2, ls='--')
        ax.set_title(f'{dataset_name} CV2: linear z0,z1\nval-fit test R2={r2:.3f}, r={r:.3f}')
        ax.set_xlabel('predicted final p-ratio')
        ax.set_ylabel('true final p-ratio')

fig.tight_layout()
plt.show()





In [ ]:
# Compact takeaway table for the current model choice.

summary_table = metrics_df.pivot_table(
    index=['dataset_name', 'latent_dim'],
    columns='readout',
    values='test_r2',
    aggfunc='first',
).reset_index()
summary_table.columns.name = None
display(summary_table.round(4))





## Initial Identity And Trajectory State Correlations

Keep the two questions separate:

1. Initial identity: `z(0)` across networks against final p-ratio and stiffness descriptors only.
2. Trajectory state: full `z(t)` trajectories against deformation variables such as box strain, box-size changes, and displacement.



In [ ]:
# Initial-frame absolute correlations: only p-ratio and stiffness descriptors.

initial_diagnostic_targets = [
    'final_p_ratio',
    'stiffness_mean',
    'stiffness_std',
    'stiffness_cv',
    'soft_frac_lt_0p2',
]

corr_rows = []
for dataset_name in cfg['dataset_names']:
    for latent_dim in cfg['latent_dims']:
        group = prediction_df[
            prediction_df['dataset_name'].eq(dataset_name)
            & prediction_df['latent_dim'].eq(latent_dim)
            & prediction_df['split'].eq('test')
        ].copy()
        if group.empty:
            continue
        coordinates = [('z0', 'z0')]
        if int(latent_dim) >= 2:
            group['linear_combo'] = group['linear_combo_pred_p_ratio']
            coordinates.extend([('z1', 'z1'), ('linear_combo', 'linear combo')])
        for coordinate, coordinate_label in coordinates:
            for target in initial_diagnostic_targets:
                signed_r = pearson_r(group[coordinate], group[target])
                corr_rows.append({
                    'dataset_name': dataset_name,
                    'latent_dim': int(latent_dim),
                    'coordinate': coordinate,
                    'coordinate_label': f'CV{int(latent_dim)} {coordinate_label}',
                    'target': target,
                    'pearson_r': signed_r,
                    'abs_pearson_r': abs(signed_r) if np.isfinite(signed_r) else np.nan,
                })

coordinate_corr_df = pd.DataFrame(corr_rows)
display(coordinate_corr_df.pivot_table(index=['dataset_name', 'coordinate_label'], columns='target', values='abs_pearson_r').round(3))




In [ ]:
# Heatmap: initial-frame absolute correlations only.

row_order = ['CV1 z0', 'CV2 z0', 'CV2 z1', 'CV2 linear combo']
for dataset_name in cfg['dataset_names']:
    heat = coordinate_corr_df[coordinate_corr_df['dataset_name'].eq(dataset_name)].pivot_table(
        index='coordinate_label', columns='target', values='abs_pearson_r'
    )
    heat = heat.reindex([row for row in row_order if row in heat.index])
    if heat.empty:
        continue
    fig, ax = plt.subplots(figsize=(6.8, 2.6))
    im = ax.imshow(heat.to_numpy(float), vmin=0, vmax=1, cmap='viridis', aspect='auto')
    ax.set_title(f'{dataset_name} initial z(0) absolute correlations')
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=25, ha='right')
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            value = heat.iloc[i, j]
            if np.isfinite(value):
                ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8, color=PLOT_COLORS['white'] if value > 0.65 else PLOT_COLORS['deep'])
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label='abs(Pearson r)')
    fig.tight_layout()
    plt.show()





## Full-Trajectory Latent Correlations

This is the dynamic question: over all frames of the held-out trajectories, does `z0(t)` or `z1(t)` track deformation state? The main targets are box-size changes, box strain, displacement, and progress. The table reports both pooled correlations and within-network demeaned correlations.



In [ ]:
# Extract full latent trajectories and deformation variables.

trajectory_rows = []
for dataset_name in cfg['dataset_names']:
    for latent_dim in cfg['latent_dims']:
        path = model_path(dataset_name, latent_dim)
        if not path.exists():
            continue
        print('trajectory eval:', path.name)
        bundle = load_bundle(path)
        ae = restore_ae(bundle)
        _, _, test_data, _ = resolve_bundle_splits(bundle)
        for sim_idx, sim in enumerate(test_data):
            max_frame = min(int(cfg['trajectory_max_frames_per_test_network']), len(sim) - 1)
            for frame_idx in range(max_frame + 1):
                z = encode_frame_z(ae, bundle, sim, frame_idx)
                row = {
                    'dataset_name': dataset_name,
                    'latent_dim': int(latent_dim),
                    'sim_idx': int(sim_idx),
                    **frame_deformation_row(sim, frame_idx),
                }
                for dim_idx, value in enumerate(z):
                    row[f'z{dim_idx}'] = float(value)
                trajectory_rows.append(row)
        del ae, bundle
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

trajectory_df = pd.DataFrame(trajectory_rows)
print('trajectory rows:', trajectory_df.shape)
display(trajectory_df.head())




## Raw CV2 z0 vs z1 Over Test Trajectories

This plots the unrotated CV2 latent coordinates for every frame from every held-out test network. Each point is one `(network, frame)` state.



In [ ]:
# Raw z0-vs-z1 scatter over all CV2 test trajectory frames.

cv2_traj = trajectory_df[trajectory_df['latent_dim'].eq(2)].copy()
if cv2_traj.empty or not {'z0', 'z1'}.issubset(cv2_traj.columns):
    print('No CV2 trajectory rows available. Run the full-trajectory extraction cell first.')
else:
    fig, ax = plt.subplots(figsize=(5.8, 5.2))
    scatter = ax.scatter(
        cv2_traj['z0'],
        cv2_traj['z1'],
        c=cv2_traj['frame_progress'],
        s=8,
        alpha=0.45,
        cmap=PLOT_CMAP,
        linewidths=0,
    )
    ax.set_xlabel('z0')
    ax.set_ylabel('z1')
    ax.set_title('de Pablo CV2 raw z0 vs z1, all test trajectory frames')
    cbar = fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('frame progress')
    ax.set_aspect('equal', adjustable='datalim')
    fig.tight_layout()
    plt.show()




## Raw CV1 z0 Over Test Trajectories

This plots the unrotated CV1 `z0(t)` coordinate for every held-out test trajectory, so we can see whether the single-CV model mostly learns a shared deformation clock or separates trajectories by network.


In [ ]:
# Raw CV1 z0(t) for all test trajectories, colored by final p-ratio.
# This recomputes z0 directly from the saved CV1 bundle, so it cannot inherit centered/shifted values from trajectory_df.

cv1_path = model_path(cfg['dataset_names'][0], 1)
if not cv1_path.exists():
    print('Missing CV1 model:', cv1_path)
else:
    cv1_bundle = load_bundle(cv1_path)
    cv1_ae = restore_ae(cv1_bundle)
    _, _, cv1_test_data, _ = resolve_bundle_splits(cv1_bundle)

    cv1_rows = []
    for sim_idx, sim in enumerate(cv1_test_data):
        final_p = p_ratio_final(sim)
        max_frame = min(int(cfg['trajectory_max_frames_per_test_network']), len(sim) - 1)
        for frame_idx in range(max_frame + 1):
            z = encode_frame_z(cv1_ae, cv1_bundle, sim, frame_idx)
            cv1_rows.append({
                'sim_idx': int(sim_idx),
                'frame_idx': int(frame_idx),
                'z0': float(z[0]),
                'final_p_ratio': final_p,
            })
    cv1_traj_raw = pd.DataFrame(cv1_rows)

    # Sanity check: frame-0 z0 here should match the initial latent table for CV1/test.
    cv1_initial = latent_df[
        latent_df['latent_dim'].eq(1) & latent_df['split'].eq('test')
    ][['sim_idx', 'z0']].rename(columns={'z0': 'z0_initial_table'})
    cv1_check = cv1_traj_raw[cv1_traj_raw['frame_idx'].eq(0)][['sim_idx', 'z0']].merge(cv1_initial, on='sim_idx', how='left')
    cv1_check['abs_diff'] = (cv1_check['z0'] - cv1_check['z0_initial_table']).abs()
    print(
        'raw CV1 z0 frame-0 max |direct - initial_table|:',
        float(cv1_check['abs_diff'].max()),
    )
    print(
        'raw CV1 z0 range:',
        float(cv1_traj_raw['z0'].min()),
        'to',
        float(cv1_traj_raw['z0'].max()),
    )
    print(
        'raw CV1 z0 first-frame range:',
        float(cv1_check['z0'].min()),
        'to',
        float(cv1_check['z0'].max()),
    )

    norm = plt.Normalize(cv1_traj_raw['final_p_ratio'].min(), cv1_traj_raw['final_p_ratio'].max())
    cmap = plt.get_cmap('coolwarm')

    fig, ax = plt.subplots(figsize=(7.4, 4.3))
    for sim_idx, group in cv1_traj_raw.groupby('sim_idx'):
        group = group.sort_values('frame_idx')
        final_p = float(group['final_p_ratio'].iloc[0])
        ax.plot(group['frame_idx'], group['z0'], lw=1.1, alpha=0.58, color=cmap(norm(final_p)))
    mean_curve = cv1_traj_raw.groupby('frame_idx', as_index=False)['z0'].mean()
    ax.plot(mean_curve['frame_idx'], mean_curve['z0'], lw=2.6, color=PLOT_COLORS['deep'], label='mean')
    ax.set_xlabel('frame')
    ax.set_ylabel('raw CV1 z0')
    ax.set_title('de Pablo CV1 raw z0(t), direct from AE, all test trajectories')
    ax.legend(frameon=False)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('final p-ratio')
    fig.tight_layout()
    plt.show()

    del cv1_ae, cv1_bundle
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Correlate full z(t) trajectories with deformation variables.
# pooled_abs_r: all networks and frames pooled.
# within_abs_r: demean z and target within each network before correlating; this isolates trajectory motion from network identity offsets.

trajectory_targets = [
    'frame_progress',
    'box_delta_width',
    'box_delta_height',
    'box_strain_x',
    'box_strain_y',
    'box_poisson_path',
    'bbox_strain_x',
    'bbox_strain_y',
    'mean_dx',
    'mean_dy',
    'rms_dx',
    'rms_dy',
    'rms_disp',
]

traj_corr_rows = []
for (dataset_name, latent_dim), group in trajectory_df.groupby(['dataset_name', 'latent_dim'], sort=False):
    coordinates = ['z0'] + (['z1'] if int(latent_dim) >= 2 and 'z1' in group.columns else [])
    for coordinate in coordinates:
        for target in trajectory_targets:
            pooled_r = pearson_r(group[coordinate], group[target])
            demeaned_parts = []
            for _, sim_group in group.groupby('sim_idx'):
                part = sim_group[[coordinate, target]].copy()
                part[coordinate] = part[coordinate] - part[coordinate].mean()
                part[target] = part[target] - part[target].mean()
                demeaned_parts.append(part)
            demeaned = pd.concat(demeaned_parts, ignore_index=True) if demeaned_parts else pd.DataFrame()
            within_r = pearson_r(demeaned[coordinate], demeaned[target]) if not demeaned.empty else np.nan
            traj_corr_rows.append({
                'dataset_name': dataset_name,
                'latent_dim': int(latent_dim),
                'coordinate': coordinate,
                'coordinate_label': f'CV{int(latent_dim)} {coordinate}',
                'target': target,
                'pooled_r': pooled_r,
                'pooled_abs_r': abs(pooled_r) if np.isfinite(pooled_r) else np.nan,
                'within_r': within_r,
                'within_abs_r': abs(within_r) if np.isfinite(within_r) else np.nan,
            })

trajectory_corr_df = pd.DataFrame(traj_corr_rows)
print('Pooled absolute correlations:')
display(trajectory_corr_df.pivot_table(index=['dataset_name', 'coordinate_label'], columns='target', values='pooled_abs_r').round(3))
print('Within-network demeaned absolute correlations:')
display(trajectory_corr_df.pivot_table(index=['dataset_name', 'coordinate_label'], columns='target', values='within_abs_r').round(3))




In [ ]:
# Heatmaps for full-trajectory correlations.

row_order = ['CV1 z0', 'CV2 z0', 'CV2 z1']
for metric, title_suffix in [('pooled_abs_r', 'pooled'), ('within_abs_r', 'within-network demeaned')]:
    for dataset_name in cfg['dataset_names']:
        heat = trajectory_corr_df[trajectory_corr_df['dataset_name'].eq(dataset_name)].pivot_table(
            index='coordinate_label', columns='target', values=metric
        )
        heat = heat.reindex([row for row in row_order if row in heat.index])
        if heat.empty:
            continue
        fig, ax = plt.subplots(figsize=(11.0, 2.3))
        im = ax.imshow(heat.to_numpy(float), vmin=0, vmax=1, cmap='viridis', aspect='auto')
        ax.set_title(f'{dataset_name} full z(t) vs deformation, {title_suffix}')
        ax.set_yticks(np.arange(len(heat.index)))
        ax.set_yticklabels(heat.index)
        ax.set_xticks(np.arange(len(heat.columns)))
        ax.set_xticklabels(heat.columns, rotation=35, ha='right')
        for i in range(heat.shape[0]):
            for j in range(heat.shape[1]):
                value = heat.iloc[i, j]
                if np.isfinite(value):
                    ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8, color=PLOT_COLORS['white'] if value > 0.65 else PLOT_COLORS['deep'])
        fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label='abs(Pearson r)')
        fig.tight_layout()
        plt.show()





## CV2 Trajectory Curvature

Some test-network paths are visibly curved in raw `z0,z1`, while others are close to a straight line. This cell quantifies curvature per network and compares it to p-ratio and deformation variables.



In [ ]:
# Per-network curvature in the raw CV2 z0-z1 trajectory.
cv2_traj = trajectory_df[trajectory_df['latent_dim'].eq(2)].copy()
if cv2_traj.empty:
    print('No CV2 trajectory rows available. Run the full-trajectory extraction cell first.')
    cv2_curvature_df = pd.DataFrame()
else:
    static_cols = ['final_p_ratio', 'stiffness_mean', 'stiffness_std', 'stiffness_cv', 'soft_frac_lt_0p2']
    static_lookup = latent_df[(latent_df['latent_dim'].eq(2)) & (latent_df['split'].eq('test'))][['sim_idx'] + static_cols].drop_duplicates('sim_idx')
    end_state = cv2_traj.sort_values('frame_idx').groupby('sim_idx', as_index=False).tail(1)[[
        'sim_idx', 'box_strain_x', 'box_strain_y', 'box_poisson_path', 'rms_dx', 'rms_dy', 'rms_disp'
    ]].rename(columns={
        'box_strain_x': 'final_box_strain_x',
        'box_strain_y': 'final_box_strain_y',
        'box_poisson_path': 'final_box_poisson_path',
        'rms_dx': 'final_rms_dx',
        'rms_dy': 'final_rms_dy',
        'rms_disp': 'final_rms_disp',
    })
    cv2_curvature_df = cv2_traj.groupby('sim_idx').apply(path_curvature_metrics, include_groups=False).reset_index()
    cv2_curvature_df = cv2_curvature_df.merge(static_lookup, on='sim_idx', how='left').merge(end_state, on='sim_idx', how='left')
    display(cv2_curvature_df.sort_values('curvature_score', ascending=False).head(12).round(5))




In [ ]:
# What differs between curved and linear trajectories?

if cv2_curvature_df.empty:
    print('No curvature rows available.')
else:
    curvature_targets = [
        'final_p_ratio',
        'stiffness_mean',
        'stiffness_std',
        'stiffness_cv',
        'soft_frac_lt_0p2',
        'final_box_strain_x',
        'final_box_strain_y',
        'final_box_poisson_path',
        'final_rms_dx',
        'final_rms_dy',
        'final_rms_disp',
    ]
    rows = []
    for metric in ['curvature_score', 'tortuosity', 'max_perp_deviation', 'mean_perp_deviation']:
        for target in curvature_targets:
            rows.append({
                'curvature_metric': metric,
                'target': target,
                'pearson_r': pearson_r(cv2_curvature_df[metric], cv2_curvature_df[target]),
                'abs_pearson_r': abs(pearson_r(cv2_curvature_df[metric], cv2_curvature_df[target])) if np.isfinite(pearson_r(cv2_curvature_df[metric], cv2_curvature_df[target])) else np.nan,
            })
    curvature_corr_df = pd.DataFrame(rows)
    display(curvature_corr_df.pivot_table(index='curvature_metric', columns='target', values='abs_pearson_r').round(3))

    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))
    plot_targets = ['final_p_ratio', 'final_box_strain_y', 'final_rms_dy']
    for ax, target in zip(axes, plot_targets):
        ax.scatter(cv2_curvature_df[target], cv2_curvature_df['curvature_score'], s=28, alpha=0.8, color=PLOT_COLORS['olive'])
        ax.set_xlabel(target)
        ax.set_ylabel('curvature score = 1 - line R2')
        ax.set_title(f'abs r={abs(pearson_r(cv2_curvature_df[target], cv2_curvature_df["curvature_score"])):.2f}')
    fig.tight_layout()
    plt.show()




In [ ]:
# Show examples: most curved vs most linear paths in raw z0-z1.

if cv2_curvature_df.empty:
    print('No curvature rows available.')
else:
    n_examples = 5
    curved_ids = cv2_curvature_df.sort_values('curvature_score', ascending=False).head(n_examples)['sim_idx'].tolist()
    linear_ids = cv2_curvature_df.sort_values('curvature_score', ascending=True).head(n_examples)['sim_idx'].tolist()
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.4), sharex=True, sharey=True)
    for ax, ids, title in zip(axes, [curved_ids, linear_ids], ['most curved', 'most linear']):
        for sim_idx in ids:
            group = cv2_traj[cv2_traj['sim_idx'].eq(sim_idx)].sort_values('frame_idx')
            meta = cv2_curvature_df[cv2_curvature_df['sim_idx'].eq(sim_idx)].iloc[0]
            ax.plot(group['z0'], group['z1'], lw=1.6, alpha=0.85, label=f"{int(sim_idx)} p={meta['final_p_ratio']:.2f} c={meta['curvature_score']:.2f}")
            ax.scatter(group['z0'].iloc[0], group['z1'].iloc[0], s=18, color=PLOT_COLORS['deep'], zorder=3)
        ax.set_title(title)
        ax.set_xlabel('z0')
        ax.set_ylabel('z1')
        ax.legend(frameon=False, fontsize=7)
    fig.suptitle('de Pablo CV2 raw trajectories: curved vs linear')
    fig.tight_layout()
    plt.show()




## Single Curved Trajectory: What Does z(t) Track?

Pick one curved CV2 trajectory, then correlate its frame-by-frame latent coordinates with frame-by-frame physical/deformation quantities. This asks what the curve means within that one trajectory, not what separates different networks.



In [ ]:
# Select one curved trajectory and add frame-wise p-ratio.
# By default: the most curved CV2 trajectory.

selected_curved_sim_idx = None  # set an int manually to inspect a specific test network

if cv2_curvature_df.empty:
    print('No curvature rows available. Run the CV2 curvature section first.')
    single_curve_df = pd.DataFrame()
else:
    if selected_curved_sim_idx is None:
        selected_curved_sim_idx = int(cv2_curvature_df.sort_values('curvature_score', ascending=False).iloc[0]['sim_idx'])
    print('selected test sim_idx:', selected_curved_sim_idx)

    cv2_path = model_path(cfg['dataset_names'][0], 2)
    cv2_bundle = load_bundle(cv2_path)
    _, _, cv2_test_data, _ = resolve_bundle_splits(cv2_bundle)
    selected_sim = cv2_test_data[int(selected_curved_sim_idx)]

    single_curve_df = cv2_traj[cv2_traj['sim_idx'].eq(selected_curved_sim_idx)].sort_values('frame_idx').copy()
    cv1_single = trajectory_df[
        trajectory_df['latent_dim'].eq(1)
        & trajectory_df['sim_idx'].eq(selected_curved_sim_idx)
    ][['frame_idx', 'z0']].rename(columns={'z0': 'cv1_z0'})
    single_curve_df = single_curve_df.merge(cv1_single, on='frame_idx', how='left')
    frame_pratios = []
    for frame_idx in single_curve_df['frame_idx'].astype(int):
        try:
            frame_pratios.append(float(calc_p_ratio_rollout_sides(selected_sim, int(frame_idx))))
        except Exception:
            frame_pratios.append(np.nan)
    single_curve_df['frame_p_ratio'] = frame_pratios
    single_curve_df['delta_z0'] = single_curve_df['z0'] - float(single_curve_df['z0'].iloc[0])
    single_curve_df['delta_z1'] = single_curve_df['z1'] - float(single_curve_df['z1'].iloc[0])

    meta = cv2_curvature_df[cv2_curvature_df['sim_idx'].eq(selected_curved_sim_idx)].iloc[0]
    print(
        f"curvature={meta['curvature_score']:.4g}, tortuosity={meta['tortuosity']:.4g}, "
        f"final_p={meta['final_p_ratio']:.4g}, final_path_poisson={meta['final_box_poisson_path']:.4g}"
    )
    display(single_curve_df.head().round(5))





In [ ]:
# Within this one trajectory: correlate CV1 z0(t), CV2 z0(t), and CV2 z1(t)
# with only the physically meaningful frame-wise quantities.

single_targets = {
    'x_deformation': 'box_strain_x',
    'y_deformation': 'box_strain_y',
    'dx_motion': 'rms_dx',
    'dy_motion': 'rms_dy',
    'per_frame_p_ratio': 'frame_p_ratio',
}

single_coordinates = {
    'CV1 z0': 'cv1_z0',
    'CV2 z0': 'z0',
    'CV2 z1': 'z1',
}

if single_curve_df.empty:
    print('No selected trajectory rows available.')
else:
    single_corr_rows = []
    for coordinate_label, coordinate in single_coordinates.items():
        if coordinate not in single_curve_df.columns:
            print(f'missing {coordinate}; rerun the selected trajectory cell after the full trajectory extraction cell')
            continue
        for signal_label, target in single_targets.items():
            signed_r = pearson_r(single_curve_df[coordinate], single_curve_df[target])
            single_corr_rows.append({
                'coordinate': coordinate_label,
                'signal': signal_label,
                'target_column': target,
                'abs_pearson_r': abs(signed_r) if np.isfinite(signed_r) else np.nan,
            })
    single_curve_corr_df = pd.DataFrame(single_corr_rows)
    print('Single curved trajectory: abs correlations with plausible signals only')
    display(single_curve_corr_df.pivot_table(index='coordinate', columns='signal', values='abs_pearson_r').round(3))


In [ ]:
# Plot the selected curved trajectory and the five plausible frame-wise signals.

if single_curve_df.empty:
    print('No selected trajectory rows available.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1))
    sc = axes[0].scatter(single_curve_df['z0'], single_curve_df['z1'], c=single_curve_df['frame_idx'], cmap=PLOT_CMAP, s=26)
    axes[0].plot(single_curve_df['z0'], single_curve_df['z1'], color=PLOT_COLORS['muted'], lw=1.2, alpha=0.7)
    axes[0].scatter(single_curve_df['z0'].iloc[0], single_curve_df['z1'].iloc[0], color=PLOT_COLORS['deep'], s=45, label='start')
    axes[0].scatter(single_curve_df['z0'].iloc[-1], single_curve_df['z1'].iloc[-1], color=PLOT_COLORS['muted'], s=45, label='end')
    axes[0].set_xlabel('z0')
    axes[0].set_ylabel('z1')
    axes[0].set_title(f'CV2 path for test sim {selected_curved_sim_idx}')
    axes[0].legend(frameon=False)
    fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label='frame')

    axes[1].plot(single_curve_df['frame_idx'], single_curve_df['z0'], label='z0', lw=2)
    axes[1].plot(single_curve_df['frame_idx'], single_curve_df['z1'], label='z1', lw=2)
    axes[1].set_xlabel('frame')
    axes[1].set_ylabel('raw latent value')
    axes[1].set_title('latent coordinates over frames')
    axes[1].legend(frameon=False)
    fig.tight_layout()
    plt.show()

    signal_cols = {
        'x_deformation': 'box_strain_x',
        'y_deformation': 'box_strain_y',
        'dx_motion': 'rms_dx',
        'dy_motion': 'rms_dy',
        'per_frame_p_ratio': 'frame_p_ratio',
    }
    fig, axes = plt.subplots(len(signal_cols), 1, figsize=(7.4, 1.75 * len(signal_cols)), sharex=True, squeeze=False)
    for ax, (label, col) in zip(axes[:, 0], signal_cols.items()):
        ax.plot(single_curve_df['frame_idx'], single_curve_df[col], color=PLOT_COLORS['olive'], lw=2)
        ax.set_ylabel(label)
        ax.set_title(label)
    axes[-1, 0].set_xlabel('frame')
    fig.suptitle('plausible frame-wise physical/deformation signals')
    fig.tight_layout()
    plt.show()




## Latent Coordinates Over Time: Auxetic Extremes

Select the two most auxetic and two least auxetic test networks by final p-ratio, then plot their latent coordinates over frames on the same axes.


In [ ]:
# Same plot: z0 and z1 over time for 2 most auxetic and 2 least auxetic test trajectories.

extreme_lookup = latent_df[
    latent_df['latent_dim'].eq(2) & latent_df['split'].eq('test')
][['sim_idx', 'final_p_ratio']].drop_duplicates('sim_idx')

if extreme_lookup.empty or trajectory_df.empty:
    print('Need latent_df and trajectory_df. Run the initial latent and full-trajectory extraction cells first.')
else:
    most_auxetic = extreme_lookup.sort_values('final_p_ratio', ascending=True).head(2).copy()
    least_auxetic = extreme_lookup.sort_values('final_p_ratio', ascending=False).head(2).copy()
    most_auxetic['group'] = 'most auxetic'
    least_auxetic['group'] = 'least auxetic'
    extreme_df = pd.concat([most_auxetic, least_auxetic], ignore_index=True)
    display(extreme_df.round(5))

    cv2_extreme = trajectory_df[
        trajectory_df['latent_dim'].eq(2)
        & trajectory_df['sim_idx'].isin(extreme_df['sim_idx'])
    ].merge(extreme_df, on='sim_idx', how='left').sort_values(['group', 'sim_idx', 'frame_idx'])

    fig, ax = plt.subplots(figsize=(9.0, 4.7), facecolor=PLOT_COLORS['soft'])
    ax.set_facecolor(PLOT_COLORS['soft'])

    coord_colors = {'z0': '#2563EB', 'z1': '#DC2626'}
    group_ls = {'most auxetic': '-', 'least auxetic': '--'}
    group_alpha = {'most auxetic': 0.95, 'least auxetic': 0.72}

    for sim_idx, group in cv2_extreme.groupby('sim_idx'):
        group = group.sort_values('frame_idx')
        kind = group['group'].iloc[0]
        final_p = float(group['final_p_ratio'].iloc[0])
        for coord in ['z0', 'z1']:
            ax.plot(
                group['frame_idx'],
                group[coord],
                lw=1.9,
                alpha=group_alpha[kind],
                color=coord_colors[coord],
                ls=group_ls[kind],
                label=f"{coord}, {kind}, sim {int(sim_idx)}, p={final_p:.2f}",
            )

    ax.set_xlabel('frame')
    ax.set_ylabel('raw latent value')
    ax.set_title('de Pablo CV2 z0 and z1 over time: auxetic extremes')
    ax.legend(frameon=False, fontsize=7, ncol=2)
    fig.tight_layout()
    plt.show()


## What Separates The Most Curved Paths?

Take the 5 most curved and 5 most linear CV2 trajectories, then ask which physical/deformation variables differ the most. This is more direct than a global correlation because it focuses on the visible phenomenon in the plot.



In [ ]:
# Compare the 5 most curved trajectories against the 5 most linear trajectories.

if cv2_curvature_df.empty:
    print('No curvature rows available.')
else:
    n_extreme = 5
    feature_cols = [
        'final_p_ratio',
        'stiffness_mean',
        'stiffness_std',
        'stiffness_cv',
        'soft_frac_lt_0p2',
        'final_box_strain_x',
        'final_box_strain_y',
        'final_box_poisson_path',
        'final_rms_dx',
        'final_rms_dy',
        'final_rms_disp',
    ]
    ranked = cv2_curvature_df.sort_values('curvature_score', ascending=False).copy()
    curved = ranked.head(n_extreme).copy()
    linear = ranked.tail(n_extreme).copy()
    curved['curvature_group'] = 'most curved'
    linear['curvature_group'] = 'most linear'
    curvature_extremes_df = pd.concat([curved, linear], ignore_index=True)

    rows = []
    for col in feature_cols:
        if col not in cv2_curvature_df.columns:
            continue
        all_std = float(cv2_curvature_df[col].std())
        curved_mean = float(curved[col].mean())
        linear_mean = float(linear[col].mean())
        diff = curved_mean - linear_mean
        rows.append({
            'feature': col,
            'curved_mean': curved_mean,
            'linear_mean': linear_mean,
            'curved_minus_linear': diff,
            'abs_standardized_diff': abs(diff) / all_std if np.isfinite(all_std) and all_std > 1e-12 else np.nan,
            'curvature_abs_r': abs(pearson_r(cv2_curvature_df['curvature_score'], cv2_curvature_df[col])),
            'curvature_signed_r': pearson_r(cv2_curvature_df['curvature_score'], cv2_curvature_df[col]),
        })
    curvature_separator_df = pd.DataFrame(rows).sort_values('abs_standardized_diff', ascending=False)

    print('Most curved trajectories:')
    display(curved[['sim_idx', 'curvature_score', 'tortuosity', 'final_p_ratio', 'final_box_poisson_path', 'final_box_strain_x', 'final_box_strain_y']].round(5))
    print('Most linear trajectories:')
    display(linear[['sim_idx', 'curvature_score', 'tortuosity', 'final_p_ratio', 'final_box_poisson_path', 'final_box_strain_x', 'final_box_strain_y']].round(5))
    print('Features that best separate most-curved from most-linear:')
    display(curvature_separator_df.round(5))




In [ ]:
# Plot the strongest separators for the extreme groups.

if 'curvature_separator_df' not in globals() or curvature_separator_df.empty:
    print('No separator table available.')
else:
    top_features = curvature_separator_df.head(4)['feature'].tolist()
    fig, axes = plt.subplots(1, len(top_features), figsize=(3.0 * len(top_features), 3.5), squeeze=False)
    for ax, feature in zip(axes[0], top_features):
        parts = [
            curvature_extremes_df[curvature_extremes_df['curvature_group'].eq('most linear')][feature].dropna(),
            curvature_extremes_df[curvature_extremes_df['curvature_group'].eq('most curved')][feature].dropna(),
        ]
        ax.boxplot(parts, labels=['linear', 'curved'], showfliers=True)
        for x_pos, vals in enumerate(parts, start=1):
            jitter = np.linspace(-0.055, 0.055, max(len(vals), 1))[:len(vals)]
            ax.scatter(np.full(len(vals), x_pos) + jitter, vals, s=22, color=PLOT_COLORS['deep'], alpha=0.7, zorder=3)
        ax.set_title(feature)
        ax.tick_params(axis='x', rotation=20)
    fig.suptitle('Most curved vs most linear CV2 paths')
    fig.tight_layout()
    plt.show()




## Testing Whether z1 Encodes Auxetic Rearrangement

To support the interpretation that CV2 `z1` is not just deformation progress, we remove the ordinary progress component and ask whether the remaining `z1` signal tracks auxetic-specific trajectory features. The key targets are the residual transverse strain after fitting `y_strain ~ x_strain`, and the instantaneous Poisson slope along the trajectory.


In [ ]:
# Build auxetic-specific framewise signals with the shared residualization helper.
aux_parts = []
cv2_traj_for_aux = trajectory_df[trajectory_df['latent_dim'].eq(2)].copy()
for sim_idx, group in cv2_traj_for_aux.groupby('sim_idx'):
    group = group.sort_values('frame_idx').copy()
    sx = group['box_strain_x'].to_numpy(float)
    sy = group['box_strain_y'].to_numpy(float)
    z0 = group['z0'].to_numpy(float)
    z1 = group['z1'].to_numpy(float)

    group['residual_y_after_x'] = residualize(sy, sx)
    group['z1_residual_after_z0'] = residualize(z1, z0)
    group['z0_residual_after_progress'] = residualize(z0, group['frame_progress'].to_numpy(float))
    group['z1_residual_after_progress'] = residualize(z1, group['frame_progress'].to_numpy(float))

    dsx = np.gradient(sx)
    dsy = np.gradient(sy)
    local = np.full_like(dsx, np.nan, dtype=float)
    mask = np.isfinite(dsx) & np.isfinite(dsy) & (np.abs(dsx) > 1e-12)
    local[mask] = -dsy[mask] / dsx[mask]
    group['local_poisson_slope'] = local
    group['local_poisson_slope_change'] = np.gradient(local) if np.isfinite(local).sum() >= 3 else np.nan

    aux_parts.append(group)

auxetic_signal_df = pd.concat(aux_parts, ignore_index=True) if aux_parts else pd.DataFrame()
print('auxetic signal rows:', auxetic_signal_df.shape)
display(auxetic_signal_df.head().round(5))


In [ ]:
# Visual proof on the selected curved trajectory: compare z1 residual with auxetic-specific signals.

if 'selected_curved_sim_idx' not in globals() or auxetic_signal_df.empty:
    print('Run the selected curved trajectory and auxetic signal cells first.')
else:
    one = auxetic_signal_df[auxetic_signal_df['sim_idx'].eq(selected_curved_sim_idx)].sort_values('frame_idx').copy()
    plot_cols = [
        ('z1 residual after z0', 'z1_residual_after_z0'),
        ('y residual after x', 'residual_y_after_x'),
        ('local Poisson slope', 'local_poisson_slope'),
        ('path Poisson', 'box_poisson_path'),
    ]
    fig, axes = plt.subplots(len(plot_cols), 1, figsize=(7.6, 1.85 * len(plot_cols)), sharex=True, squeeze=False)
    for ax, (label, col) in zip(axes[:, 0], plot_cols):
        values = one[col].to_numpy(float)
        if np.nanstd(values) > 1e-12:
            values = (values - np.nanmean(values)) / np.nanstd(values)
        ax.plot(one['frame_idx'], values, lw=2, color=PLOT_COLORS['deep'] if 'z1' in label else PLOT_COLORS['muted'])
        ax.set_ylabel(label)
    axes[-1, 0].set_xlabel('frame')
    fig.suptitle(f'Selected curved sim {selected_curved_sim_idx}: normalized auxetic-mode signals')
    fig.tight_layout()
    plt.show()



## Most Auxetic Trajectories: z1 Residual vs Transverse Residual

For the most auxetic test networks, compare the part of CV2 `z1` not explained by `z0` against `residual_y_after_x`, the y-deformation left after regressing out x-deformation within the same trajectory.


In [ ]:
# Plot the strongest concrete signal found for z1 residual on one auxetic trajectory.
# sim 31 is the most auxetic trajectory in the current de Pablo test split.

selected_auxetic_sim_idx = 31

if 'auxetic_signal_df' not in globals() or auxetic_signal_df.empty:
    print('Run the auxetic signal cell first.')
else:
    one = auxetic_signal_df[auxetic_signal_df['sim_idx'].eq(selected_auxetic_sim_idx)].sort_values('frame_idx').copy()
    if one.empty:
        print(f'No rows found for sim {selected_auxetic_sim_idx}. Available sims:', sorted(auxetic_signal_df['sim_idx'].unique())[:20])
    else:
        pratio_col = next((c for c in ['final_p_ratio', 'final_pratio', 'p_ratio', 'pratio'] if c in one.columns), None)
        if pratio_col is not None:
            final_p = float(one[pratio_col].iloc[0])
        else:
            lookup = latent_df[latent_df['latent_dim'].eq(2)][['sim_idx', 'final_p_ratio']].drop_duplicates('sim_idx')
            final_p = float(lookup.loc[lookup['sim_idx'].eq(selected_auxetic_sim_idx), 'final_p_ratio'].iloc[0])

        x = one['frame_idx'].to_numpy(float)
        z_res = one['z1_residual_after_z0'].to_numpy(float)
        y_res = one['residual_y_after_x'].to_numpy(float)

        def standardize(values):
            values = np.asarray(values, dtype=float)
            scale = np.nanstd(values)
            if not np.isfinite(scale) or scale < 1e-12:
                return values * np.nan
            return (values - np.nanmean(values)) / scale

        z_plot = standardize(z_res)
        y_plot = standardize(y_res)
        corr = pearson_r(z_res, y_res)

        fig, ax = plt.subplots(figsize=(8.0, 3.8))
        ax.plot(x, z_plot, lw=2.4, color='#364735', label='CV2 z1 residual after CV2 z0')
        ax.plot(x, y_plot, lw=2.2, color='#626456', ls='--', label='y-deformation residual after x-deformation')
        ax.axhline(0.0, color='#545B4C', lw=0.8, alpha=0.35)
        ax.set_facecolor(PLOT_COLORS['soft'])
        ax.set_xlabel('trajectory frame')
        ax.set_ylabel('standardized residual value')
        ax.set_title(f'de Pablo sim {selected_auxetic_sim_idx}: z1 residual tracks transverse residual deformation\nfinal p-ratio={final_p:.3f}, abs correlation={abs(corr):.3f}')
        ax.legend(frameon=False, fontsize=9)
        ax.grid(True, alpha=0.22)
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        fig.tight_layout()
        plt.show()

